# Notebook 07 — StratLake Feature Consumption, Baseline Research Smoke Test, and Archive Checkpoint

Standalone continuation notebook for the Fintech → StratLake Colab workflow.

This revision emphasizes runnability:

- installs the project packages,
- mounts Google Drive,
- initializes or reconnects both sessions,
- uses padded market-data and feature-build windows around Q1,
- validates that daily bars and features exist before consumption,
- filters the research smoke test to the Q1 analysis window after warmup data has been generated,
- keeps archive/session outputs session-scoped and non-canonical.

## Install notebook dependencies and project packages

These are kept here so the notebook can run independently in a fresh Colab runtime.

In [ ]:
!pip install "pandas-market-calendars>=5.0"
!pip install -i https://test.pypi.org/simple/ fintech-market-ingestion
!pip install -i https://test.pypi.org/simple/ stratlake-trade-engine

## Verify expected CLIs

In [ ]:
import shutil

required_commands = [
    "fintech-init-project",
    "fintech-backfill-daily",
    "fintech-save-session",
    "fintech-restore-session",
    "fintech-backup-data",
    "stratlake-init-session",
    "stratlake-build-features",
    "stratlake-session-export",
    "stratlake-session-import",
    "stratlake-session-archive-bootstrap",
    "stratlake-session-archive-restore-bootstrap",
]

optional_strategy_commands = [
    "stratlake-run-strategy",
    "stratlake-backtest",
    "stratlake-run-backtest",
    "stratlake-compare-strategies",
]

missing_commands = []
for command in required_commands:
    path = shutil.which(command)
    print(f"{command}: {path if path else 'NOT FOUND'}")
    if path is None:
        missing_commands.append(command)

print("Optional strategy/backtest commands:")
for command in optional_strategy_commands:
    path = shutil.which(command)
    print(f"{command}: {path if path else 'not installed / not exposed as CLI'}")

if missing_commands:
    raise RuntimeError("Missing required commands: " + ", ".join(missing_commands))

## Imports and Google Drive mount

In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
from datetime import datetime, timezone

import pandas as pd

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
else:
    print("Not running in Colab; skipping Google Drive mount.")

## Configure workspace and tutorial roots

These defaults match Notebook 04–06. Override the names or session IDs below when reconnecting to a previous run.

In [ ]:
# Colab-safe workspace roots.
# In Colab, keep active work under /content and use Drive only for persistence/archive packs.
WORKSPACE_ROOT = Path("/content") if IN_COLAB else Path.cwd()
DRIVE_FOLDER_NAME = "REPLACE_WITH_DRIVE_FOLDER_NAME"
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME if IN_COLAB else WORKSPACE_ROOT / "drive" / DRIVE_FOLDER_NAME

if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    raise ValueError("Set DRIVE_FOLDER_NAME before creating Google Drive session/archive folders.")

FINTECH_ROOT = WORKSPACE_ROOT / "fintech-market-ingestion-demo"
STRATLAKE_ROOT = WORKSPACE_ROOT / "stratlake-trade-engine-demo"

FINTECH_DRIVE_ROOT = DRIVE_ROOT / "fintech-market-ingestion"
STRATLAKE_DRIVE_ROOT = DRIVE_ROOT / "stratlake-trade-engine"

FINTECH_SESSION_NAME = "fintech_stratlake_input"
STRATLAKE_SESSION_NAME = "stratlake_q1_feature_consumption"

# Optional: set these to reconnect Notebook 07 to an existing Notebook 05/06 run.
FINTECH_SESSION_ID_OVERRIDE = ""
STRATLAKE_SESSION_ID_OVERRIDE = ""

# Q1 research target plus padded build/ingestion windows.
# The warmup window gives rolling/lagged feature builders data before Q1 so early-Q1 rows are less likely to be NaN.
ANALYSIS_START = "2026-01-02"
ANALYSIS_END = "2026-03-31"

BACKFILL_START = "2025-11-03"   # warmup before Q1
BACKFILL_END = "2026-04-15"     # buffer after Q1 for forward-return checks

FEATURE_BUILD_START = BACKFILL_START
FEATURE_BUILD_END = BACKFILL_END

BACKFILL_SYMBOLS = "AAPL,MSFT,NVDA,SPY,QQQ"

for path in [FINTECH_ROOT, STRATLAKE_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("WORKSPACE_ROOT:", WORKSPACE_ROOT)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("FINTECH_ROOT:", FINTECH_ROOT)
print("STRATLAKE_ROOT:", STRATLAKE_ROOT)
print("ANALYSIS window:", ANALYSIS_START, "to", ANALYSIS_END)
print("Padded backfill/build window:", BACKFILL_START, "to", BACKFILL_END)

## Initialize or attach to the Fintech session

`fintech-init-project` currently accepts `--session-name` and `--with-session`; it does **not** accept StratLake-style `--project-name`, `--drive-root`, or `--enable-drive-persistence` flags. Google Drive persistence is handled by the notebook path variables below.

In [ ]:
if FINTECH_SESSION_ID_OVERRIDE:
    FINTECH_SESSION_ID = FINTECH_SESSION_ID_OVERRIDE
    print("Using overridden FINTECH_SESSION_ID:", FINTECH_SESSION_ID)
else:
    # Correct current fintech-market-ingestion initialization pattern.
    # Drive persistence is notebook-managed through DRIVE_ROOT / session-scoped paths.
    !fintech-init-project \
      --root {FINTECH_ROOT.as_posix()} \
      --session-name {FINTECH_SESSION_NAME} \
      --with-session \
      --colab-profile

    FINTECH_SESSION_ID = None

    # Prefer the generated session manifest when available.
    session_dir = FINTECH_ROOT / "artifacts" / "sessions"
    if session_dir.exists():
        manifests = sorted(
            session_dir.glob("*/session_manifest.json"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        if manifests:
            manifest = json.loads(manifests[0].read_text(encoding="utf-8"))
            FINTECH_SESSION_ID = manifest.get("session_id") or manifests[0].parent.name
            print("FINTECH_SESSION_MANIFEST:", manifests[0])

    # Fallback for package versions that write a single project/session file.
    if FINTECH_SESSION_ID is None:
        session_file = FINTECH_ROOT / ".fintech" / "session.json"
        if session_file.exists():
            manifest = json.loads(session_file.read_text(encoding="utf-8"))
            FINTECH_SESSION_ID = (
                manifest.get("session_id")
                or manifest.get("session_name")
                or manifest.get("project_name")
                or FINTECH_SESSION_NAME
            )
            print("FINTECH_SESSION_FILE:", session_file)

    if FINTECH_SESSION_ID is None:
        FINTECH_SESSION_ID = FINTECH_SESSION_NAME
        print("Could not discover a generated Fintech session id; falling back to session name.")

print("FINTECH_SESSION_ID:", FINTECH_SESSION_ID)

## Initialize or attach to the StratLake session with notebook configs

In [ ]:
MARKETLAKE_ROOT = FINTECH_ROOT / "data" / "curated"
MARKETLAKE_ROOT.mkdir(parents=True, exist_ok=True)

if STRATLAKE_SESSION_ID_OVERRIDE:
    STRATLAKE_SESSION_ID = STRATLAKE_SESSION_ID_OVERRIDE
    print("Using overridden STRATLAKE_SESSION_ID:", STRATLAKE_SESSION_ID)
else:
    !stratlake-init-session \
      --root {STRATLAKE_ROOT.as_posix()} \
      --project-name {STRATLAKE_SESSION_NAME} \
      --marketlake-root {MARKETLAKE_ROOT.as_posix()} \
      --drive-root {DRIVE_ROOT.as_posix()} \
      --enable-drive-persistence \
      --notebook-configs

    stratlake_session_file = STRATLAKE_ROOT / ".stratlake" / "session.json"
    if stratlake_session_file.exists():
        manifest = json.loads(stratlake_session_file.read_text(encoding="utf-8"))
        STRATLAKE_SESSION_ID = manifest.get("session_id") or manifest.get("project_name") or STRATLAKE_SESSION_NAME
        print("STRATLAKE_SESSION_FILE:", stratlake_session_file)
    else:
        STRATLAKE_SESSION_ID = STRATLAKE_SESSION_NAME
        print("Could not discover a generated StratLake session id; falling back to project name.")

print("MARKETLAKE_ROOT:", MARKETLAKE_ROOT)
print("STRATLAKE_SESSION_ID:", STRATLAKE_SESSION_ID)

## Build session-scoped Drive paths and archive identifiers

In [ ]:
FINTECH_DRIVE_SESSIONS_ROOT = FINTECH_DRIVE_ROOT / "sessions"
STRATLAKE_DRIVE_SESSIONS_ROOT = STRATLAKE_DRIVE_ROOT / "sessions"

FINTECH_DRIVE_SESSION_ROOT = FINTECH_DRIVE_SESSIONS_ROOT / FINTECH_SESSION_ID
STRATLAKE_DRIVE_SESSION_ROOT = STRATLAKE_DRIVE_SESSIONS_ROOT / STRATLAKE_SESSION_ID

FINTECH_DRIVE_BACKUP_ROOT = FINTECH_DRIVE_SESSION_ROOT / "backups"
STRATLAKE_DRIVE_ARCHIVE_ROOT = STRATLAKE_DRIVE_SESSION_ROOT / "archives"

FINTECH_ARCHIVE_ID = f"curated-data-{FINTECH_SESSION_ID}"
STRATLAKE_ARCHIVE_ID = f"stratlake-session-{STRATLAKE_SESSION_ID}"

FINTECH_BACKUP_PACK_DIR = FINTECH_DRIVE_BACKUP_ROOT / FINTECH_ARCHIVE_ID
STRATLAKE_ARCHIVE_PACK_DIR = STRATLAKE_DRIVE_ARCHIVE_ROOT / STRATLAKE_ARCHIVE_ID

if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    raise ValueError(
        "Set DRIVE_FOLDER_NAME before creating Google Drive session/archive folders."
    )

for path in [
    FINTECH_DRIVE_SESSION_ROOT,
    STRATLAKE_DRIVE_SESSION_ROOT,
    FINTECH_DRIVE_BACKUP_ROOT,
    STRATLAKE_DRIVE_ARCHIVE_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

# Canonical local handoff locations.
MARKETLAKE_ROOT = FINTECH_ROOT / "data" / "curated"
DAILY_BARS_ROOT = MARKETLAKE_ROOT / "bars_daily"
FEATURES_DAILY_ROOT = STRATLAKE_ROOT / "data" / "curated" / "features_daily"

MARKETLAKE_ROOT.mkdir(parents=True, exist_ok=True)
DAILY_BARS_ROOT.mkdir(parents=True, exist_ok=True)

FINTECH_ROOT_STR = FINTECH_ROOT.resolve().as_posix()
STRATLAKE_ROOT_STR = STRATLAKE_ROOT.resolve().as_posix()
MARKETLAKE_ROOT_STR = MARKETLAKE_ROOT.resolve().as_posix()
DAILY_BARS_ROOT_STR = DAILY_BARS_ROOT.resolve().as_posix()
FEATURES_DAILY_ROOT_STR = FEATURES_DAILY_ROOT.as_posix()
DRIVE_ROOT_STR = DRIVE_ROOT.as_posix()
FINTECH_DRIVE_BACKUP_ROOT_STR = FINTECH_DRIVE_BACKUP_ROOT.as_posix()
STRATLAKE_DRIVE_SESSION_ROOT_STR = STRATLAKE_DRIVE_SESSION_ROOT.as_posix()
STRATLAKE_DRIVE_ARCHIVE_ROOT_STR = STRATLAKE_DRIVE_ARCHIVE_ROOT.as_posix()

for name, value in [
    ("FINTECH_SESSION_ID", FINTECH_SESSION_ID),
    ("STRATLAKE_SESSION_ID", STRATLAKE_SESSION_ID),
    ("FINTECH_ARCHIVE_ID", FINTECH_ARCHIVE_ID),
    ("STRATLAKE_ARCHIVE_ID", STRATLAKE_ARCHIVE_ID),
    ("MARKETLAKE_ROOT", MARKETLAKE_ROOT_STR),
    ("DAILY_BARS_ROOT", DAILY_BARS_ROOT_STR),
    ("FEATURES_DAILY_ROOT", FEATURES_DAILY_ROOT_STR),
    ("FINTECH_DRIVE_SESSION_ROOT", FINTECH_DRIVE_SESSION_ROOT),
    ("STRATLAKE_DRIVE_SESSION_ROOT", STRATLAKE_DRIVE_SESSION_ROOT),
    ("FINTECH_BACKUP_PACK_DIR", FINTECH_BACKUP_PACK_DIR),
    ("STRATLAKE_ARCHIVE_PACK_DIR", STRATLAKE_ARCHIVE_PACK_DIR),
]:
    print(f"{name}: {value}")

## Configure Alpaca credentials for standalone recovery/backfill

In [ ]:
import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None

def get_secret_or_prompt(name: str) -> str:
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass.getpass(f"Enter {name}: ")
    return value

alpaca_api_key_id = get_secret_or_prompt("ALPACA_API_KEY_ID")
alpaca_api_secret_key = get_secret_or_prompt("ALPACA_API_SECRET_KEY")

if not alpaca_api_key_id or not alpaca_api_secret_key:
    raise ValueError("Missing Alpaca API credentials.")

os.environ["ALPACA_API_KEY_ID"] = alpaca_api_key_id
os.environ["ALPACA_API_SECRET_KEY"] = alpaca_api_secret_key
os.environ["ALPACA_DATA_BASE_URL"] = "https://data.alpaca.markets"
os.environ["ALPACA_FEED"] = "iex"

print("ALPACA_DATA_BASE_URL:", os.environ.get("ALPACA_DATA_BASE_URL"))
print("ALPACA_FEED:", os.environ.get("ALPACA_FEED"))
print("ALPACA_API_KEY_ID and ALPACA_API_SECRET_KEY are set but not printed.")

## Optional: restore previous Fintech and StratLake archives before consuming features

In [ ]:
RESTORE_FINTECH_SESSION_ID = FINTECH_SESSION_ID
RESTORE_STRATLAKE_SESSION_ID = STRATLAKE_SESSION_ID

RESTORE_FINTECH_ARCHIVE_ID = f"curated-data-{RESTORE_FINTECH_SESSION_ID}"
RESTORE_STRATLAKE_ARCHIVE_ID = f"stratlake-session-{RESTORE_STRATLAKE_SESSION_ID}"

RESTORE_FINTECH_DRIVE_BACKUP_ROOT = FINTECH_DRIVE_SESSIONS_ROOT / RESTORE_FINTECH_SESSION_ID / "backups"
RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT = STRATLAKE_DRIVE_SESSIONS_ROOT / RESTORE_STRATLAKE_SESSION_ID / "archives"

fintech_restore_cmd = [
    "fintech-backup-data",
    "restore",
    "--root", FINTECH_ROOT_STR,
    "--archive-id", RESTORE_FINTECH_ARCHIVE_ID,
    "--backup-root", RESTORE_FINTECH_DRIVE_BACKUP_ROOT.as_posix(),
    "--copy-policy", "overwrite_allowed",
    "--validate-after-copy",
    "--inspect-after-copy",
]

stratlake_restore_cmd = [
    "stratlake-session-archive-restore-bootstrap",
    "--root", STRATLAKE_ROOT_STR,
    "--archive-id", RESTORE_STRATLAKE_ARCHIVE_ID,
    "--drive-root", RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT.as_posix(),
    "--copy-policy", "overwrite_allowed",
    "--include-features",
    "--include-artifacts",
    "--include-configs",
    "--validate-after-copy",
    "--inspect-after-copy",
]

print("Fintech restore preview:")
print(" \\\n  ".join(fintech_restore_cmd))

print("\nStratLake restore preview:")
print(" \\\n  ".join(stratlake_restore_cmd))

RESTORE_FINTECH_ARCHIVE = False
RESTORE_STRATLAKE_ARCHIVE = False

if RESTORE_FINTECH_ARCHIVE:
    subprocess.run(fintech_restore_cmd, check=True)
if RESTORE_STRATLAKE_ARCHIVE:
    subprocess.run(stratlake_restore_cmd, check=True)

## Verify notebook config files and session portability

In [ ]:
config_paths = [
    STRATLAKE_ROOT / "configs" / "universe.yml",
    STRATLAKE_ROOT / "configs" / "paths.yml",
]

for config_path in config_paths:
    print(config_path, "exists:", config_path.exists())
    if config_path.exists():
        text = config_path.read_text(encoding="utf-8", errors="replace")
        print("  size:", len(text), "chars")
        if "marketlake" in text.lower() or str(MARKETLAKE_ROOT) in text:
            print("  appears to reference MarketLake / curated data settings")

missing_configs = [p for p in config_paths if not p.exists()]
if missing_configs:
    raise FileNotFoundError("Missing required notebook config files: " + ", ".join(str(p) for p in missing_configs))

## Discover Fintech curated daily bars

In [ ]:
from pathlib import Path
import pandas as pd


def extract_partition_value(path: Path, key: str):
    """
    Extract Hive-style partition value from paths like:
      .../symbol=AAPL/date=2025-11-03/file.parquet
    """
    for part in path.parts:
        if part.startswith(f"{key}="):
            return part.split("=", 1)[1]
    return None


def extract_date_coverage_from_parquets(files: list[Path], max_files: int = 500) -> pd.DataFrame:
    rows = []

    for p in list(files)[:max_files]:
        p = Path(p)

        path_symbol = extract_partition_value(p, "symbol")
        path_date = extract_partition_value(p, "date")

        try:
            df = pd.read_parquet(p)
            read_error = None
        except Exception as exc:
            df = pd.DataFrame()
            read_error = str(exc)

        lower_to_original = {str(c).lower(): c for c in df.columns}

        date_col = next(
            (
                lower_to_original[c]
                for c in ["date", "timestamp", "datetime", "bar_time", "time"]
                if c in lower_to_original
            ),
            None,
        )

        symbol_col = next(
            (
                lower_to_original[c]
                for c in ["symbol", "ticker", "asset", "asset_id"]
                if c in lower_to_original
            ),
            None,
        )

        # Prefer dataframe date column if present; otherwise use path partition date.
        if date_col is not None and not df.empty:
            dates = pd.to_datetime(df[date_col], errors="coerce").dropna()
            min_date = dates.min() if not dates.empty else pd.NaT
            max_date = dates.max() if not dates.empty else pd.NaT
            date_source = f"column:{date_col}"
        elif path_date:
            min_date = pd.to_datetime(path_date, errors="coerce")
            max_date = pd.to_datetime(path_date, errors="coerce")
            date_source = "path:date"
        else:
            min_date = pd.NaT
            max_date = pd.NaT
            date_source = None

        # Prefer dataframe symbol column if present; otherwise use path partition symbol.
        if symbol_col is not None and not df.empty:
            symbols = ",".join(sorted(map(str, df[symbol_col].dropna().unique()))[:10])
            symbol_source = f"column:{symbol_col}"
        else:
            symbols = path_symbol
            symbol_source = "path:symbol" if path_symbol else None

        rows.append({
            "path": p.as_posix(),
            "rows": len(df),
            "columns": len(df.columns),
            "date_source": date_source,
            "symbol_source": symbol_source,
            "symbol": symbols,
            "min_date": min_date,
            "max_date": max_date,
            "read_error": read_error,
        })

    coverage = pd.DataFrame(rows)

    if not coverage.empty:
        coverage["min_date"] = pd.to_datetime(coverage["min_date"], errors="coerce")
        coverage["max_date"] = pd.to_datetime(coverage["max_date"], errors="coerce")

    return coverage

In [ ]:
def file_inventory(root: Path, patterns=("*.parquet",), limit: int | None = None) -> tuple[pd.DataFrame, list[Path]]:
    root = Path(root)
    files: list[Path] = []
    if root.exists():
        for pattern in patterns:
            files.extend(root.rglob(pattern))
    files = sorted({p for p in files if p.is_file()})
    rows = []
    for p in files if limit is None else files[:limit]:
        try:
            rel = p.relative_to(root).as_posix()
        except ValueError:
            rel = p.name
        rows.append({
            "path": p.as_posix(),
            "relative_path": rel,
            "size_bytes": p.stat().st_size,
            "modified_utc": datetime.fromtimestamp(p.stat().st_mtime, tz=timezone.utc).isoformat(),
        })
    return pd.DataFrame(rows), files


daily_bars_summary, daily_bar_files = file_inventory(MARKETLAKE_ROOT, patterns=("*.parquet",), limit=50)
print(f"Daily-bar parquet files under MARKETLAKE_ROOT: {len(daily_bar_files)}")
display(daily_bars_summary.head(50))

if daily_bar_files:
    daily_coverage = extract_date_coverage_from_parquets(daily_bar_files, max_files=80)
    display(daily_coverage.head(30))
    if "min_date" in daily_coverage.columns:
        print("Observed sampled daily-bar date range:", daily_coverage["min_date"].min(), "to", daily_coverage["max_date"].max())
else:
    print("No local daily-bar parquet files were found. Use the padded backfill cell below or restore a Fintech archive.")

## Optional: padded daily-bars backfill if curated data is missing

This cell backfills before and after Q1 so rolling features and forward-return checks have warmup/buffer data.

In [ ]:
RUN_SMALL_DAILY_BARS_BACKFILL = True
FORCE_DAILY_BARS_BACKFILL = False

FINTECH_CONFIGS_ROOT = FINTECH_ROOT / "configs"
FINTECH_CONFIGS_ROOT.mkdir(parents=True, exist_ok=True)

FINTECH_TICKERS_FILE = FINTECH_CONFIGS_ROOT / "tickers_sample.txt"
FINTECH_TICKERS_FILE.write_text(
    "\n".join(s.strip() for s in BACKFILL_SYMBOLS.split(",") if s.strip()) + "\n",
    encoding="utf-8",
)

FINTECH_TICKERS_FILE_STR = FINTECH_TICKERS_FILE.as_posix()
FINTECH_BACKFILL_SOURCE = (
    FINTECH_SESSION_ID
    if str(FINTECH_SESSION_ID).startswith("session_")
    else f"session_{FINTECH_SESSION_ID}"
)

print("FINTECH_TICKERS_FILE:", FINTECH_TICKERS_FILE_STR)
print(FINTECH_TICKERS_FILE.read_text(encoding="utf-8"))
print("FINTECH_BACKFILL_SOURCE:", FINTECH_BACKFILL_SOURCE)
print("Padded backfill window:", BACKFILL_START, "to", BACKFILL_END)
print("Daily bars output:", DAILY_BARS_ROOT_STR)

should_backfill = FORCE_DAILY_BARS_BACKFILL or not daily_bar_files

if RUN_SMALL_DAILY_BARS_BACKFILL and should_backfill:
    !fintech-backfill-daily \
      --symbols {FINTECH_TICKERS_FILE_STR} \
      --start {BACKFILL_START} \
      --end {BACKFILL_END} \
      --out {DAILY_BARS_ROOT_STR} \
      --feed iex \
      --source {FINTECH_BACKFILL_SOURCE} \
      --window month
elif RUN_SMALL_DAILY_BARS_BACKFILL:
    print("Daily bars already exist. Set FORCE_DAILY_BARS_BACKFILL=True to refresh them.")
else:
    print("Backfill skipped. Set RUN_SMALL_DAILY_BARS_BACKFILL=True only if local curated data is missing.")

# Refresh inventory after optional backfill.
daily_bars_summary, daily_bar_files = file_inventory(MARKETLAKE_ROOT, patterns=("*.parquet",), limit=50)
print(f"Daily-bar parquet files after optional backfill: {len(daily_bar_files)}")
display(daily_bars_summary.head(50))

In [ ]:
# Validate that padded daily bars cover the Q1 analysis window with warmup/buffer data.
if daily_bar_files:
    daily_coverage = extract_date_coverage_from_parquets(daily_bar_files, max_files=100)
    display(daily_coverage.head(30))
    if {"min_date", "max_date"}.issubset(daily_coverage.columns):
        observed_min = daily_coverage["min_date"].min()
        observed_max = daily_coverage["max_date"].max()
        print("Sampled daily-bar date range:", observed_min, "to", observed_max)
        print("Required padded window:", BACKFILL_START, "to", BACKFILL_END)
        print("Q1 analysis window:", ANALYSIS_START, "to", ANALYSIS_END)
else:
    print("No daily bars available yet.")

## Discover StratLake feature outputs

In [ ]:
def resolve_existing_root(name: str, candidates: list[Path]) -> Path:
    checked = []
    for root in candidates:
        root = Path(root).expanduser()
        checked.append(root)
        if root.exists():
            return root.resolve()
    print(f"Could not resolve {name}. Candidates checked:")
    for root in checked:
        print(" -", root.as_posix(), "| exists:", root.exists())
    raise FileNotFoundError(f"Could not resolve {name}.")

STRATLAKE_ROOT = resolve_existing_root(
    "STRATLAKE_ROOT",
    [
        Path(STRATLAKE_ROOT),
        Path(STRATLAKE_ROOT_STR),
        Path("/content") / "stratlake-trade-engine-demo",
        Path.cwd() / "stratlake-trade-engine-demo",
    ],
)
STRATLAKE_ROOT_STR = STRATLAKE_ROOT.as_posix()
FEATURES_DAILY_ROOT = STRATLAKE_ROOT / "data" / "curated" / "features_daily"

feature_candidate_roots = [
    FEATURES_DAILY_ROOT,
    STRATLAKE_ROOT / "data" / "curated",
    STRATLAKE_ROOT / "data",
    STRATLAKE_ROOT / "features",
    STRATLAKE_ROOT / "artifacts",
]

print("Resolved STRATLAKE_ROOT:", STRATLAKE_ROOT_STR)
print("Expected FEATURES_DAILY_ROOT:", FEATURES_DAILY_ROOT.as_posix())
print("\nCandidate feature roots:")
for root in feature_candidate_roots:
    print(" -", root.as_posix(), "| exists:", root.exists())

feature_rows = []
feature_files = []

for root in feature_candidate_roots:
    inventory, files = file_inventory(root, patterns=("*.parquet",), limit=None)
    if not inventory.empty:
        inventory["discovery_root"] = root.as_posix()
        feature_rows.append(inventory)
        feature_files.extend(files)

feature_inventory = (
    pd.concat(feature_rows, ignore_index=True)
    .drop_duplicates(subset=["path"])
    .sort_values("path")
    if feature_rows
    else pd.DataFrame()
)

feature_files = [Path(p) for p in feature_inventory["path"].tolist()] if not feature_inventory.empty else []

print("\nFeature parquet candidates:", len(feature_files))
if feature_inventory.empty:
    print("No feature parquet outputs were found yet.")
    data_root = STRATLAKE_ROOT / "data"
    if data_root.exists():
        print("\nDirectory tree under STRATLAKE_ROOT/data:")
        for p in sorted(data_root.rglob("*"))[:200]:
            kind = "DIR " if p.is_dir() else "FILE"
            print(f"{kind}: {p.as_posix()}")
else:
    display(feature_inventory.head(100))

## Optional: build StratLake features if outputs are missing

This uses the same padded window as ingestion, then later analysis is filtered back to Q1.

In [ ]:
print("STRATLAKE_ROOT_STR:", STRATLAKE_ROOT_STR)
print("MARKETLAKE_ROOT_STR:", MARKETLAKE_ROOT_STR)
print("Feature build window:", FEATURE_BUILD_START, "to", FEATURE_BUILD_END)
print("Q1 analysis window:", ANALYSIS_START, "to", ANALYSIS_END)

In [ ]:
RUN_FEATURE_BUILD_IF_MISSING = True
FORCE_FEATURE_BUILD = False

STRATLAKE_CONFIGS_ROOT = STRATLAKE_ROOT / "configs"
STRATLAKE_CONFIGS_ROOT.mkdir(parents=True, exist_ok=True)

STRATLAKE_TICKERS_FILE = STRATLAKE_CONFIGS_ROOT / "tickers_sample.txt"
STRATLAKE_TICKERS_FILE.write_text(
    "\n".join(s.strip() for s in BACKFILL_SYMBOLS.split(",") if s.strip()) + "\n",
    encoding="utf-8",
)
STRATLAKE_TICKERS_FILE_STR = STRATLAKE_TICKERS_FILE.as_posix()

feature_candidates_before = feature_files
print("Feature parquet files before optional build:", len(feature_candidates_before))
print("STRATLAKE_TICKERS_FILE:", STRATLAKE_TICKERS_FILE_STR)
print(STRATLAKE_TICKERS_FILE.read_text(encoding="utf-8"))

should_build_features = FORCE_FEATURE_BUILD or not feature_candidates_before

if RUN_FEATURE_BUILD_IF_MISSING and should_build_features:
    build_cmd = [
        "stratlake-build-features",
        "--timeframe", "1D",
        "--start", FEATURE_BUILD_START,
        "--end", FEATURE_BUILD_END,
        "--tickers", STRATLAKE_TICKERS_FILE_STR,
        "--marketlake-root", MARKETLAKE_ROOT_STR,
    ]
    print("Running feature build from:", STRATLAKE_ROOT_STR)
    print("Command:")
    print(" ".join(build_cmd))
    subprocess.run(build_cmd, cwd=STRATLAKE_ROOT_STR, check=True)
elif RUN_FEATURE_BUILD_IF_MISSING:
    print("Feature outputs already exist. Set FORCE_FEATURE_BUILD=True to rebuild.")
else:
    print("Feature build skipped.")

# Refresh feature inventory after optional build.
feature_rows = []
feature_files = []
for root in feature_candidate_roots:
    inventory, files = file_inventory(root, patterns=("*.parquet",), limit=None)
    if not inventory.empty:
        inventory["discovery_root"] = root.as_posix()
        feature_rows.append(inventory)
        feature_files.extend(files)

feature_inventory = (
    pd.concat(feature_rows, ignore_index=True)
    .drop_duplicates(subset=["path"])
    .sort_values("path")
    if feature_rows
    else pd.DataFrame()
)
feature_files = [Path(p) for p in feature_inventory["path"].tolist()] if not feature_inventory.empty else []

print("Feature parquet files after optional build:", len(feature_files))
display(feature_inventory.head(100) if not feature_inventory.empty else pd.DataFrame())

## Load a representative daily-bars sample and feature sample

In [ ]:
def load_parquet_sample(files: list[Path], label: str, max_files: int = 80) -> pd.DataFrame:
    frames = []
    for p in files[:max_files]:
        try:
            df = pd.read_parquet(p)
            if df.empty:
                continue
            df = df.copy()
            df["_source_path"] = p.as_posix()
            frames.append(df)
        except Exception as exc:
            print(f"Could not read {label} file {p}: {exc}")
    if not frames:
        print(f"No usable {label} parquet files found.")
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True, sort=False)
    print(f"Loaded {label} sample from {len(frames)} files; shape={out.shape}")
    return out

sample_daily_df = load_parquet_sample(daily_bar_files, "daily bars", max_files=100)
sample_feature_df = load_parquet_sample(feature_files, "features", max_files=100)

print("Daily bars columns:")
print(list(sample_daily_df.columns)[:60])
print("Feature columns:")
print(list(sample_feature_df.columns)[:100])

display(sample_daily_df.head())
display(sample_feature_df.head())

## Prepare loaded parquet samples for coverage diagnostics

This section is intentionally limited to inspection and coverage reporting. Notebook 07 uses the native StratLake CLI for the baseline strategy smoke test when `configs/strategies.yml` is available. The lightweight dataframe normalization below only supports notebook diagnostics and fallback plotting; it is not a replacement for native Fintech or StratLake commands.


In [ ]:
# Diagnostic-only normalization for loaded parquet samples.
# Native StratLake CLI commands remain the authoritative strategy/signal path.

def normalize_symbol_date_frame(df: pd.DataFrame, label: str) -> pd.DataFrame:
    if df.empty:
        print(f"{label}: no rows loaded.")
        return df.copy()

    out = df.copy()
    lower_to_original = {str(c).lower(): c for c in out.columns}

    symbol_col = next(
        (lower_to_original[c] for c in ["symbol", "ticker", "asset", "asset_id"] if c in lower_to_original),
        None,
    )
    date_col = next(
        (lower_to_original[c] for c in ["date", "timestamp", "datetime", "bar_time", "time"] if c in lower_to_original),
        None,
    )

    rename_map = {}
    if symbol_col and symbol_col != "symbol":
        rename_map[symbol_col] = "symbol"
    if date_col and date_col != "date":
        rename_map[date_col] = "date"
    if rename_map:
        out = out.rename(columns=rename_map)
        print(f"{label}: diagnostic rename map:", rename_map)

    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.tz_localize(None).dt.normalize()
    if "symbol" in out.columns:
        out["symbol"] = out["symbol"].astype(str).str.upper()

    return out

bars = normalize_symbol_date_frame(sample_daily_df, "daily bars")
features = normalize_symbol_date_frame(sample_feature_df, "features")

analysis_start_dt = pd.Timestamp(ANALYSIS_START)
analysis_end_dt = pd.Timestamp(ANALYSIS_END)

bars_analysis = (
    bars[(bars["date"] >= analysis_start_dt) & (bars["date"] <= analysis_end_dt)].copy()
    if {"date"}.issubset(bars.columns)
    else pd.DataFrame()
)
features_analysis = (
    features[(features["date"] >= analysis_start_dt) & (features["date"] <= analysis_end_dt)].copy()
    if {"date"}.issubset(features.columns)
    else pd.DataFrame()
)

print("Bars columns:", list(bars.columns)[:60])
print("Features columns:", list(features.columns)[:100])
print("Bars have symbol/date:", {"symbol", "date"}.issubset(bars.columns))
print("Features have symbol/date:", {"symbol", "date"}.issubset(features.columns))

if "date" in bars.columns and not bars.empty:
    print("Full bars date range:", bars["date"].min(), "to", bars["date"].max())
    print("Q1 bars rows:", len(bars_analysis))

if "date" in features.columns and not features.empty:
    print("Full features date range:", features["date"].min(), "to", features["date"].max())
    print("Q1 feature rows:", len(features_analysis))

    numeric_feature_cols = [
        c for c in features_analysis.select_dtypes(include="number").columns
        if str(c).lower() not in {"open", "high", "low", "close", "adj_close", "volume", "trade_count", "vwap"}
    ]
    if numeric_feature_cols:
        nan_report = (
            features_analysis[numeric_feature_cols]
            .isna()
            .mean()
            .rename("nan_fraction")
            .reset_index()
            .rename(columns={"index": "feature"})
            .sort_values("nan_fraction")
        )
        print("Q1 feature NaN report:")
        display(nan_report.head(25))
    else:
        print("No numeric feature columns found for NaN coverage reporting.")


## Native StratLake CLI strategy smoke test

This is the preferred Notebook 07 baseline validation path. It uses the installed StratLake command and generated notebook configs instead of recreating strategy logic in ad hoc notebook code.

Expected native inputs:

- `configs/strategies.yml`
- `configs/universe.yml`
- `configs/paths.yml`
- `data/curated/features_daily`

The notebook-local feature/forward-return diagnostic below is retained only as a fallback when the native CLI strategy path is unavailable.


In [ ]:
from pathlib import Path
import os
import re
import subprocess
import json
import pandas as pd

RUN_NATIVE_BASELINE_SMOKE = True
NATIVE_STRATEGY_NAME = "momentum_v1"

# Use the Q1 analysis window for strategy evaluation. The padded window remains for ingestion/build warmup.
STRATEGY_START = ANALYSIS_START
STRATEGY_END = ANALYSIS_END

native_smoke_completed = False
native_strategy_stdout = ""
native_strategy_stderr = ""
native_strategy_returncode = None
native_strategy_metrics = {}
smoke_result = pd.DataFrame()
smoke_timeseries = pd.DataFrame()

# Resolve roots.
STRATLAKE_ROOT = Path(STRATLAKE_ROOT)
if not STRATLAKE_ROOT.is_absolute():
    candidate = Path("/content") / STRATLAKE_ROOT
    STRATLAKE_ROOT = candidate.resolve() if candidate.exists() else STRATLAKE_ROOT.resolve()
STRATLAKE_ROOT_STR = STRATLAKE_ROOT.as_posix()

MARKETLAKE_ROOT = Path(MARKETLAKE_ROOT)
if not MARKETLAKE_ROOT.is_absolute():
    candidate = Path("/content") / MARKETLAKE_ROOT
    MARKETLAKE_ROOT = candidate.resolve() if candidate.exists() else MARKETLAKE_ROOT.resolve()
MARKETLAKE_ROOT_STR = MARKETLAKE_ROOT.as_posix()

os.environ["STRATLAKE_ROOT"] = STRATLAKE_ROOT_STR
os.environ["MARKETLAKE_ROOT"] = MARKETLAKE_ROOT_STR

print("STRATLAKE_ROOT:", STRATLAKE_ROOT_STR)
print("MARKETLAKE_ROOT:", MARKETLAKE_ROOT_STR)

# The CLI pattern expects configs/strategies.yml relative to the StratLake workspace root.
os.chdir(STRATLAKE_ROOT)
print("Current working directory:", Path.cwd().as_posix())

STRATEGIES_CONFIG = Path("configs") / "strategies.yml"
required_paths = [
    Path("configs") / "universe.yml",
    Path("configs") / "paths.yml",
    STRATEGIES_CONFIG,
    Path("data") / "curated" / "features_daily",
]

print("\nRequired native strategy inputs:")
for p in required_paths:
    print(" -", p.as_posix(), "| exists:", p.exists())


def parse_native_strategy_stdout(stdout: str) -> dict:
    """Parse the stable human-readable StratLake CLI smoke output into one summary row."""
    metrics: dict[str, object] = {}
    patterns = {
        "strategy": r"^strategy:\s*(.+)$",
        "run_id": r"^run_id:\s*(.+)$",
        "cumulative_return": r"^cumulative_return:\s*([-+0-9.eE]+)",
        "sharpe_ratio": r"^sharpe_ratio:\s*([-+0-9.eE]+)",
        "qa_status": r"^- status:\s*(.+)$",
        "qa_rows_symbols": r"^- rows:\s*(\d+)\s*\|\s*symbols:\s*(\d+)",
        "trades_turnover": r"^- trades:\s*(\d+)\s*\|\s*turnover:\s*([-+0-9.eE]+)",
        "avg_holding": r"^- avg holding:\s*([-+0-9.eE]+)",
        "benchmark_return_pct": r"^- benchmark return:\s*([-+0-9.eE]+)%",
        "excess_return_pct": r"^- excess return:\s*([-+0-9.eE]+)%",
        "correlation": r"^- correlation:\s*([-+0-9.eE]+)",
    }

    for line in stdout.splitlines():
        stripped = line.strip()
        for key, pattern in patterns.items():
            match = re.search(pattern, stripped)
            if not match:
                continue
            if key == "qa_rows_symbols":
                metrics["qa_rows"] = int(match.group(1))
                metrics["qa_symbols"] = int(match.group(2))
            elif key == "trades_turnover":
                metrics["trades"] = int(match.group(1))
                metrics["turnover"] = float(match.group(2))
            elif key.endswith("_pct"):
                metrics[key.replace("_pct", "")] = float(match.group(1)) / 100.0
            elif key in {"cumulative_return", "sharpe_ratio", "avg_holding", "correlation"}:
                metrics[key] = float(match.group(1))
            else:
                metrics[key] = match.group(1)

        signal_match = re.search(
            r"^- long:\s*(\d+)%\s*\|\s*short:\s*(\d+)%\s*\|\s*flat:\s*(\d+)%",
            stripped,
        )
        if signal_match:
            metrics["long_fraction"] = int(signal_match.group(1)) / 100.0
            metrics["short_fraction"] = int(signal_match.group(2)) / 100.0
            metrics["flat_fraction"] = int(signal_match.group(3)) / 100.0

    metrics.setdefault("strategy", NATIVE_STRATEGY_NAME)
    metrics["start"] = STRATEGY_START
    metrics["end"] = STRATEGY_END
    metrics["source"] = "native_stratlake_cli"
    return metrics


def discover_native_strategy_timeseries(root: Path, run_id: str | None) -> pd.DataFrame:
    """Best-effort inspection of native Strategy artifacts for plotting.

    This does not recreate strategy logic. It only loads native artifacts if a time-series file exists.
    """
    if not run_id:
        return pd.DataFrame()

    artifact_root = root / "artifacts"
    if not artifact_root.exists():
        return pd.DataFrame()

    candidates = []
    for pattern in ["*.parquet", "*.csv", "*.json", "*.jsonl"]:
        candidates.extend(artifact_root.rglob(pattern))

    candidates = [p for p in candidates if run_id in p.as_posix()]
    preferred_value_cols = [
        "cumulative_return",
        "equity_curve",
        "portfolio_value",
        "portfolio_return",
        "daily_return",
        "return",
    ]

    for p in sorted(candidates):
        try:
            if p.suffix == ".parquet":
                df = pd.read_parquet(p)
            elif p.suffix == ".csv":
                df = pd.read_csv(p)
            elif p.suffix in {".json", ".jsonl"}:
                df = pd.read_json(p, lines=(p.suffix == ".jsonl"))
            else:
                continue
        except Exception:
            continue

        if df.empty:
            continue

        lower_to_original = {str(c).lower(): c for c in df.columns}
        date_col = next((lower_to_original[c] for c in ["date", "timestamp", "datetime", "time"] if c in lower_to_original), None)
        value_col = next((lower_to_original[c] for c in preferred_value_cols if c in lower_to_original), None)
        if not date_col or not value_col:
            continue

        out = df[[date_col, value_col]].copy().rename(columns={date_col: "date", value_col: "value"})
        out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.tz_localize(None).dt.normalize()
        out["value"] = pd.to_numeric(out["value"], errors="coerce")
        out = out.dropna(subset=["date", "value"]).sort_values("date")
        if out.empty:
            continue

        if str(value_col).lower() in {"daily_return", "return", "portfolio_return"}:
            out["cumulative_return"] = (1.0 + out["value"]).cumprod() - 1.0
        elif str(value_col).lower() == "portfolio_value":
            first_value = out["value"].iloc[0]
            out["cumulative_return"] = out["value"] / first_value - 1.0 if first_value else pd.NA
        else:
            out["cumulative_return"] = out["value"]

        out["artifact_path"] = p.as_posix()
        out["source"] = "native_strategy_artifact"
        return out[["date", "cumulative_return", "artifact_path", "source"]].dropna(subset=["cumulative_return"])

    return pd.DataFrame()


if RUN_NATIVE_BASELINE_SMOKE and STRATEGIES_CONFIG.exists():
    cmd = [
        "stratlake-run-strategy",
        "--strategies-config", STRATEGIES_CONFIG.as_posix(),
        "--strategy", NATIVE_STRATEGY_NAME,
        "--start", STRATEGY_START,
        "--end", STRATEGY_END,
    ]

    print("\nRunning native StratLake strategy command:")
    print(" ".join(cmd))

    result = subprocess.run(
        cmd,
        cwd=STRATLAKE_ROOT,
        text=True,
        capture_output=True,
    )

    native_strategy_stdout = result.stdout or ""
    native_strategy_stderr = result.stderr or ""
    native_strategy_returncode = result.returncode

    print("\nSTDOUT:")
    print(native_strategy_stdout)
    if native_strategy_stderr:
        print("\nSTDERR:")
        print(native_strategy_stderr)

    native_strategy_metrics = parse_native_strategy_stdout(native_strategy_stdout)
    native_strategy_metrics["returncode"] = native_strategy_returncode

    if result.returncode == 0:
        native_smoke_completed = True
        print("\nNative StratLake strategy smoke test completed.")
        smoke_result = pd.DataFrame([native_strategy_metrics])
        smoke_timeseries = discover_native_strategy_timeseries(
            STRATLAKE_ROOT,
            native_strategy_metrics.get("run_id"),
        )
    else:
        print("\nNative StratLake strategy smoke test failed.")
        print("Return code:", result.returncode)
        print("Falling back to notebook-local feature/forward-return diagnostic.")
else:
    print("\nSkipping native StratLake strategy smoke test.")
    if not RUN_NATIVE_BASELINE_SMOKE:
        print("Reason: RUN_NATIVE_BASELINE_SMOKE is False.")
    elif not STRATEGIES_CONFIG.exists():
        print("Reason: configs/strategies.yml does not exist in the StratLake workspace.")
        print("Expected path:", (STRATLAKE_ROOT / STRATEGIES_CONFIG).as_posix())
        print("Falling back to notebook-local feature/forward-return diagnostic.")

RUN_NOTEBOOK_LOCAL_FALLBACK_SMOKE = not native_smoke_completed
print("\nRUN_NOTEBOOK_LOCAL_FALLBACK_SMOKE:", RUN_NOTEBOOK_LOCAL_FALLBACK_SMOKE)

if not smoke_result.empty:
    print("\nSmoke-test output rows:")
    display(smoke_result)

if not smoke_timeseries.empty:
    print("Native strategy time-series rows discovered:", len(smoke_timeseries))
    display(smoke_timeseries.head())
else:
    print("No native strategy time-series artifact was discovered. Plot cell will use summary metrics.")


In [ ]:
# ---------------------------------------------------------------------
# Native strategy smoke-test outcome
# ---------------------------------------------------------------------

native_strategy_status = {
    "strategy": NATIVE_STRATEGY_NAME,
    "start": STRATEGY_START,
    "end": STRATEGY_END,
    "completed": native_smoke_completed,
    "returncode": native_strategy_returncode,
    "fallback_needed": RUN_NOTEBOOK_LOCAL_FALLBACK_SMOKE,
    "run_id": native_strategy_metrics.get("run_id") if native_strategy_metrics else None,
    "qa_status": native_strategy_metrics.get("qa_status") if native_strategy_metrics else None,
    "qa_rows": native_strategy_metrics.get("qa_rows") if native_strategy_metrics else None,
    "qa_symbols": native_strategy_metrics.get("qa_symbols") if native_strategy_metrics else None,
}

native_strategy_status_df = pd.DataFrame([native_strategy_status])
display(native_strategy_status_df)

if native_smoke_completed:
    print("Native StratLake CLI strategy smoke test completed successfully.")
    print("Notebook-local fallback diagnostic is not needed.")
else:
    print("Native StratLake CLI strategy smoke test did not complete.")
    print("Notebook-local fallback diagnostic should run.")


## Fallback-only notebook-local diagnostic

This cell runs only when the native StratLake strategy CLI is unavailable or fails. It is intentionally labeled as a diagnostic, not a native strategy implementation. When the native CLI succeeds, `smoke_result` is already populated from native command output and this cell does not overwrite it.


In [ ]:
# ---------------------------------------------------------------------
# Fallback-only feature/forward-return merge diagnostic
# ---------------------------------------------------------------------
# This avoids replacing native StratLake behavior with notebook logic.
# It only checks whether loaded feature rows can be joined to daily bars.
# ---------------------------------------------------------------------

if not RUN_NOTEBOOK_LOCAL_FALLBACK_SMOKE:
    print("Skipping notebook-local diagnostic because native StratLake CLI strategy smoke test completed.")

else:
    def find_close_column(df: pd.DataFrame):
        lower_to_original = {str(c).lower(): c for c in df.columns}
        for candidate in ["close", "adj_close", "close_price", "price"]:
            if candidate in lower_to_original:
                return lower_to_original[candidate]
        return None

    fallback_result = pd.DataFrame()

    if (
        not bars.empty
        and not features_analysis.empty
        and {"symbol", "date"}.issubset(bars.columns)
        and {"symbol", "date"}.issubset(features_analysis.columns)
    ):
        close_col = find_close_column(bars)

        excluded = {
            "open", "high", "low", "close", "adj_close", "volume", "trade_count", "vwap",
            "forward_return_1d", "feature_rank_pct",
        }
        candidate_features = [
            c for c in features_analysis.select_dtypes(include="number").columns
            if str(c).lower() not in excluded
        ]

        feature_col = None
        for c in candidate_features:
            s = features_analysis[c]
            if s.notna().sum() >= 10 and s.nunique(dropna=True) > 1:
                feature_col = c
                break

        if close_col and feature_col:
            bars_sorted = bars[["symbol", "date", close_col]].dropna().sort_values(["symbol", "date"])
            bars_sorted["forward_return_1d"] = bars_sorted.groupby("symbol")[close_col].pct_change().shift(-1)

            feature_frame = features_analysis[["symbol", "date", feature_col]].dropna().copy()
            merged = feature_frame.merge(
                bars_sorted[["symbol", "date", "forward_return_1d"]],
                on=["symbol", "date"],
                how="inner",
            )
            merged = merged.dropna(subset=[feature_col, "forward_return_1d"])

            if not merged.empty:
                merged["feature_rank_pct"] = merged.groupby("date")[feature_col].rank(pct=True)
                merged["long_bucket"] = merged["feature_rank_pct"] >= 0.8
                merged["short_bucket"] = merged["feature_rank_pct"] <= 0.2

                daily = merged.groupby("date", group_keys=False).apply(
                    lambda x: pd.Series({
                        "long_mean_forward_return": x.loc[x["long_bucket"], "forward_return_1d"].mean(),
                        "short_mean_forward_return": x.loc[x["short_bucket"], "forward_return_1d"].mean(),
                        "n_obs": len(x),
                        "n_long": int(x["long_bucket"].sum()),
                        "n_short": int(x["short_bucket"].sum()),
                    })
                ).reset_index()

                daily["long_short_spread"] = daily["long_mean_forward_return"] - daily["short_mean_forward_return"]
                daily["source"] = "notebook_local_fallback_diagnostic"
                daily["strategy"] = "feature_rank_fallback"
                daily["feature_column"] = feature_col
                fallback_result = daily

                smoke_result = fallback_result
                smoke_timeseries = fallback_result[["date", "long_short_spread"]].copy()
                smoke_timeseries["cumulative_return"] = smoke_timeseries["long_short_spread"].fillna(0.0).cumsum()
                smoke_timeseries["source"] = "notebook_local_fallback_diagnostic"

                print("Fallback diagnostic feature column:", feature_col)
                print("Merged Q1 observations:", len(merged))
                print("Fallback diagnostic date range:", smoke_result["date"].min(), "to", smoke_result["date"].max())
                display(smoke_result.head(20))
            else:
                print("No overlapping Q1 feature/bar rows after merge.")
        else:
            print("Could not find both a close price column and a usable numeric feature column.")
            print("close_col:", close_col)
            print("candidate_features:", candidate_features[:20])
    else:
        print("Skipping fallback diagnostic because bars/features are empty or missing symbol/date columns.")

    if fallback_result.empty:
        print("Fallback diagnostic produced no rows.")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# ---------------------------------------------------------------------
# Plot native or fallback smoke-test output
# ---------------------------------------------------------------------
# Default behavior now works in both cases:
#   1. Plot a native strategy time series if a native artifact was discovered.
#   2. Otherwise plot a native summary metrics bar chart.
#   3. Otherwise plot fallback cumulative long/short spread.
# ---------------------------------------------------------------------

if "smoke_timeseries" in globals() and isinstance(smoke_timeseries, pd.DataFrame) and not smoke_timeseries.empty:
    plot_df = smoke_timeseries.copy()
    plot_df["date"] = pd.to_datetime(plot_df["date"], errors="coerce")
    plot_df = plot_df.dropna(subset=["date"]).sort_values("date")

    if "cumulative_return" in plot_df.columns and plot_df["cumulative_return"].notna().any():
        ax = plot_df.set_index("date")["cumulative_return"].plot(
            title="Smoke test: cumulative return / cumulative diagnostic series",
            figsize=(10, 4),
            marker="o" if len(plot_df) <= 5 else None,
        )
        ax.set_xlabel("Date")
        ax.set_ylabel("Cumulative return / spread")
        ax.grid(True, alpha=0.3)
        plt.show()
        display(plot_df.tail(20))
    else:
        print("smoke_timeseries exists, but no cumulative_return column was available.")

elif "smoke_result" in globals() and isinstance(smoke_result, pd.DataFrame) and not smoke_result.empty:
    # Native CLI often returns a single summary row instead of a daily time series.
    # Plot available summary metrics so the cell remains useful by default.
    metric_cols = [
        c for c in ["cumulative_return", "sharpe_ratio", "benchmark_return", "excess_return", "correlation", "turnover"]
        if c in smoke_result.columns and pd.api.types.is_numeric_dtype(smoke_result[c])
    ]

    if metric_cols:
        summary_plot = smoke_result.iloc[0][metric_cols].astype(float)
        ax = summary_plot.plot(
            kind="bar",
            title="Native StratLake strategy smoke-test summary metrics",
            figsize=(10, 4),
        )
        ax.set_xlabel("Metric")
        ax.set_ylabel("Value")
        ax.grid(True, axis="y", alpha=0.3)
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()
        display(smoke_result)
    elif "long_short_spread" in smoke_result.columns:
        plot_df = smoke_result.copy()
        plot_df["date"] = pd.to_datetime(plot_df["date"], errors="coerce")
        plot_df = plot_df.dropna(subset=["date"]).sort_values("date")
        plot_df["cumulative_long_short_spread"] = pd.to_numeric(
            plot_df["long_short_spread"],
            errors="coerce",
        ).fillna(0.0).cumsum()
        ax = plot_df.set_index("date")["cumulative_long_short_spread"].plot(
            title="Notebook-local fallback diagnostic: cumulative long/short spread",
            figsize=(10, 4),
        )
        ax.set_xlabel("Date")
        ax.set_ylabel("Cumulative spread")
        ax.grid(True, alpha=0.3)
        plt.show()
        display(plot_df.tail(20))
    else:
        print("smoke_result exists, but no plottable numeric summary metrics were found.")
        display(smoke_result)
else:
    print("No smoke-test output rows are available to plot.")
    print("Check the native CLI cell and fallback diagnostic cell above.")


## Strategy/backtest command discovery

Use this to locate package-level strategy commands when the installed StratLake version exposes them. Notebook 07 keeps the package workflow separate from the lightweight notebook smoke test above.

In [ ]:
strategy_command_candidates = [
    "stratlake-run-strategy",
    "stratlake-backtest",
    "stratlake-run-backtest",
    "python -m src.cli.run_strategy",
    "python -m src.cli.backtest",
]

for candidate in strategy_command_candidates:
    executable = candidate.split()[0]
    path = shutil.which(executable) if executable != "python" else shutil.which("python")
    print(f"{candidate}: {'available' if path else 'not available'}")

print("When a package-level strategy command is available, prefer it for formal research artifacts.")
print("This notebook-level smoke test is only a handoff validation step.")

## Export StratLake session to Drive, dry run

In [ ]:
export_cmd = [
    "stratlake-session-export",
    "--root", STRATLAKE_ROOT_STR,
    "--drive-root", STRATLAKE_DRIVE_SESSION_ROOT_STR,
    "--include-features",
    "--include-artifacts",
    "--include-configs",
    "--dry-run",
]

print("StratLake session export dry-run command:")
print(" ".join(export_cmd))

try:
    subprocess.run(export_cmd, check=True)
except FileNotFoundError:
    print("stratlake-session-export command was not found in this package version.")
except subprocess.CalledProcessError as exc:
    print("Dry-run export command returned a non-zero status:", exc.returncode)

## Optional StratLake archive checkpoint after feature consumption

In [ ]:
stratlake_archive_cmd = [
    "stratlake-session-archive-bootstrap",
    "--root", STRATLAKE_ROOT_STR,
    "--archive-id", STRATLAKE_ARCHIVE_ID,
    "--archive-collision-policy", "overwrite_allowed",
    "--drive-root", STRATLAKE_DRIVE_ARCHIVE_ROOT_STR,
    "--copy-policy", "overwrite_allowed",
    "--include-features",
    "--include-artifacts",
    "--include-configs",
    "--validate-after-copy",
    "--inspect-after-copy",
]

print("StratLake archive checkpoint command:")
print(" \
  ".join(stratlake_archive_cmd))

CREATE_STRATLAKE_ARCHIVE_AFTER_CONSUMPTION = False

if CREATE_STRATLAKE_ARCHIVE_AFTER_CONSUMPTION:
    subprocess.run(stratlake_archive_cmd, check=True)
else:
    print("Preview only. Set CREATE_STRATLAKE_ARCHIVE_AFTER_CONSUMPTION=True to create the archive checkpoint.")

## Final Notebook 07 handoff summary

In [ ]:
summary = {
    "fintech_session_id": FINTECH_SESSION_ID,
    "stratlake_session_id": STRATLAKE_SESSION_ID,
    "analysis_start": ANALYSIS_START,
    "analysis_end": ANALYSIS_END,
    "padded_backfill_start": BACKFILL_START,
    "padded_backfill_end": BACKFILL_END,
    "feature_build_start": FEATURE_BUILD_START,
    "feature_build_end": FEATURE_BUILD_END,
    "marketlake_root": MARKETLAKE_ROOT_STR,
    "daily_bars_root": DAILY_BARS_ROOT_STR,
    "features_daily_root": FEATURES_DAILY_ROOT.as_posix(),
    "fintech_archive_id": FINTECH_ARCHIVE_ID,
    "stratlake_archive_id": STRATLAKE_ARCHIVE_ID,
    "fintech_backup_pack_dir": FINTECH_BACKUP_PACK_DIR.as_posix(),
    "stratlake_archive_pack_dir": STRATLAKE_ARCHIVE_PACK_DIR.as_posix(),
    "daily_bar_file_count": len(daily_bar_files),
    "feature_file_count": len(feature_files),
    "q1_bars_rows_loaded": len(bars_analysis) if "bars_analysis" in globals() else None,
    "q1_feature_rows_loaded": len(features_analysis) if "features_analysis" in globals() else None,
    "native_strategy_completed": bool(native_smoke_completed) if "native_smoke_completed" in globals() else None,
    "native_strategy_run_id": native_strategy_metrics.get("run_id") if "native_strategy_metrics" in globals() and native_strategy_metrics else None,
    "native_strategy_qa_status": native_strategy_metrics.get("qa_status") if "native_strategy_metrics" in globals() and native_strategy_metrics else None,
    "smoke_result_rows": len(smoke_result) if "smoke_result" in globals() and isinstance(smoke_result, pd.DataFrame) else None,
    "smoke_timeseries_rows": len(smoke_timeseries) if "smoke_timeseries" in globals() and isinstance(smoke_timeseries, pd.DataFrame) else None,
    "configs_present": all(path.exists() for path in config_paths),
}

print(json.dumps(summary, indent=2, default=str))

print("\nRecommended next notebook:")
print("Notebook 08 — Formal StratLake strategy/backtest artifacts using package APIs/CLI")
